# Episode 6 — Basis swaps

Companion notebook for the video. Basis swaps exchange one floating rate for another. We use two of them to build the whole AUD curve family from one AONIA curve: AONIA/BBSW basis swaps give the 3-month BBSW curve, and 3s6s basis swaps give the 6-month BBSW curve. Then we price a 3s6s basis swap by hand and measure its basis risk. Every number on screen or in the narration is produced here and read from `build/outputs.json`.

> **Illustrative data.** The quotes in `quotes_illustrative.csv` are made up for teaching (the AONIA quotes are Episode 3's). They are **not market prices**. Educational material only, not investment advice.

1. Setup · 2. Conventions · 3. Quotes · 4. Building the curve family · 5. Forwards · 6. A 3s6s basis swap by hand · 7. What the curves say swaps should be · 8. Basis risk · 9. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-22"
quotes_file = "quotes_illustrative.csv"
output_json = "build/outputs.json"
notional = 100_000_000
basis_tenor = "5Y"

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
iso = lambda d: d.ISO()
cal = ql.Australia(ql.Australia.Settlement)
# QuantLib 1.43 misses NSW's additional Anzac Day holidays when 25 April falls on a weekend.
for d in [ql.Date(27, 4, 2026), ql.Date(26, 4, 2027)]:
    cal.addHoliday(d)
spot = cal.advance(today, 1, ql.Days)

out = {"meta": {
    "episode": 6, "valuation_date": valuation_date, "quantlib_version": ql.__version__,
    "quotes_label": "ILLUSTRATIVE - not market data",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
print("QuantLib", ql.__version__, "| valuation date", today)

The curve-building code shared by Episodes 6 to 9:

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

## 2. Conventions

| Item | Convention | Source |
|---|---|---|
| 3s6s basis swap | 3-month BBSW paid quarterly vs 6-month BBSW paid semi-annually | AFMA §2.3 |
| AONIA/BBSW basis swap ("BOB") | AONIA compounded vs BBSW; quarterly against 3-month BBSW | AFMA §2.3, §3.17 |
| The spread | quoted on the shorter leg: 3-month BBSW in a 3s6s, AONIA in a BOB | AFMA §3.6 |
| Start | T+1 | AFMA §3.6 |
| Customary size, 3s6s | about AUD 40,000 of risk per basis point | AFMA §3.4 |

Modelling note: AFMA §5.2 says BOB settlements are paid two business days after maturity; QuantLib's `OvernightIborBasisSwapRateHelper` has no payment lag, so our BOB helpers ignore it. The effect on the curve is a fraction of a basis point.

## 3. Quotes (illustrative)

In [ ]:
quotes = pd.read_csv(quotes_file)
out["quotes"] = quotes.to_dict("records")
basis = quotes[quotes.instrument.str.contains("basis")].pivot(index="tenor", columns="curve", values="value")
basis = basis.loc[["1Y", "2Y", "3Y", "5Y", "7Y", "10Y"]].reset_index()
out["basis_table"] = [{"tenor": r.tenor, "bob_bp": r.BBSW3M, "b36_bp": r.BBSW6M} for r in basis.itertuples()]
fix = quotes[quotes.instrument == "fixing"].set_index("curve").value
out["fixings"] = {"bbsw3m_pct": float(fix["BBSW3M"]), "bbsw6m_pct": float(fix["BBSW6M"])}
basis

## 4. Building the curve family

1. **AONIA** from the OIS quotes, exactly as in Episode 3. It discounts everything.
2. **3-month BBSW**: today's fixing, then AONIA/BBSW basis swaps. The AONIA leg is known from step 1, so each quote pins down the BBSW forwards.
3. **6-month BBSW**: today's fixing, then 3s6s basis swaps. The 3-month leg is known from step 2.

In [ ]:
curves, H, quote_handles, helpers = build_curves(quotes)
rep = []
for curve, hs in helpers.items():
    for h in hs:
        rep.append({"curve": curve, "maturity": iso(h.maturityDate()),
                    "error_bp": (h.impliedQuote() - h.quote().value()) * 1e4})
rep = pd.DataFrame(rep)
out["reprice_max_abs_error_bp"] = float(rep.error_bp.abs().max())
print("largest repricing error (bp):", out["reprice_max_abs_error_bp"])

## 5. Forwards

Each index at its own tenor, starting monthly: 3-month compounded AONIA, 3-month BBSW, 6-month BBSW.

In [ ]:
def fwd(curve, d0, months):
    d1 = cal.advance(d0, months, ql.Months, ql.ModifiedFollowing, False)
    return curve.forwardRate(d0, d1, dc, ql.Simple).rate() * 100

grid = []
for m in range(0, 115):  # 6-month windows that end within the 10-year curve
    d0 = cal.advance(today, m, ql.Months)
    grid.append({"t_years": dc.yearFraction(today, d0), "aonia3m_pct": fwd(curves["AONIA"], d0, 3),
                 "bbsw3m_pct": fwd(curves["BBSW3M"], d0, 3), "bbsw6m_pct": fwd(curves["BBSW6M"], d0, 6)})
g = pd.DataFrame(grid)
out["forwards"] = {"grid": g.to_dict("records"),
                   "ticks": [{"x": float(y), "label": "0" if y == 0 else f"{y}Y"} for y in range(0, 10)]}
fig, ax = plt.subplots(figsize=(9, 3.5))
for col, lab in [("bbsw6m_pct", "6M BBSW"), ("bbsw3m_pct", "3M BBSW"), ("aonia3m_pct", "AONIA (3M compounded)")]:
    ax.plot(g.t_years, g[col], label=lab)
ax.set(xlabel="years", ylabel="%"); ax.grid(alpha=.3); ax.legend()

## 6. A 3s6s basis swap by hand

Receive 6-month BBSW semi-annually; pay 3-month BBSW **plus a spread** $s$ quarterly. Both legs are discounted on AONIA. At the par spread the legs are worth the same:

$$\sum_j N \tau_j F^{6M}_j P(t_j) = \sum_i N \tau_i \left(F^{3M}_i + s\right) P(t_i) \quad\Rightarrow\quad s = \frac{\text{PV}_{6M} - \text{PV}_{3M}}{N \cdot A_{3M}}$$

where $A_{3M} = \sum_i \tau_i P(t_i)$ is the annuity of the quarterly leg.

In [ ]:
aonia = curves["AONIA"]
end = cal.advance(spot, ql.Period(basis_tenor), ql.ModifiedFollowing, False)

def leg_pv(months, curve):
    sched = ql.Schedule(spot, end, ql.Period(months, ql.Months), cal, ql.ModifiedFollowing, ql.ModifiedFollowing,
                        ql.DateGeneration.Forward, False)
    pv, annuity = 0.0, 0.0
    for d0, d1 in zip(list(sched)[:-1], list(sched)[1:]):
        tau = dc.yearFraction(d0, d1)
        f = (curve.discount(d0) / curve.discount(d1) - 1) / tau
        pv += notional * tau * f * aonia.discount(d1)
        annuity += tau * aonia.discount(d1)
    return pv, annuity, len(sched) - 1

pv6, _, n6 = leg_pv(6, curves["BBSW6M"])
pv3, a3, n3 = leg_pv(3, curves["BBSW3M"])
spread_hand = (pv6 - pv3) / (notional * a3)
h5 = next(h for h in helpers["BBSW6M"] if iso(h.maturityDate()) == iso(end))
b = {"tenor": basis_tenor, "notional": notional, "pv_6m_leg": pv6, "pv_3m_leg": pv3, "annuity_3m": a3,
     "n_6m": n6, "n_3m": n3, "spread_hand_bp": spread_hand * 1e4, "spread_ql_bp": h5.impliedQuote() * 1e4,
     "quote_bp": quote_handles[("BBSW6M", basis_tenor)].value() * 1e4}
out["basis_swap"] = b
pd.Series(b)

## 7. What the curves say swaps should be

Par rates of vanilla swaps, using AFMA's standard: quarterly against 3-month BBSW out to three years, semi-annual against 6-month BBSW from four years.

In [ ]:
par = []
for tenor, m in [("1Y", 3), ("2Y", 3), ("3Y", 3), ("4Y", 6), ("5Y", 6), ("7Y", 6), ("10Y", 6)]:
    fc = H["BBSW3M"] if m == 3 else H["BBSW6M"]
    s = vanilla_swap(tenor, 0.04, m, fc, H["AONIA"], notional)
    par.append({"tenor": tenor, "floating": f"{m}M BBSW, " + ("quarterly" if m == 3 else "semi-annual"),
                "par_pct": s.fairRate() * 100})
out["par_swaps"] = par
pd.DataFrame(par)

## 8. Basis risk

The 5-year 3s6s swap at its par spread. Bump one group of quotes at a time by +1bp, let the curves rebuild, and reprice. A basis swap has almost no outright rate risk: its risk is to the basis.

AFMA's customary 3s6s parcel carries about AUD 40,000 per basis point of risk (§3.4); we back out the matching notional.

In [ ]:
s3 = ql.Schedule(spot, end, ql.Period(ql.Quarterly), cal, ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
s6 = ql.Schedule(spot, end, ql.Period(ql.Semiannual), cal, ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
spread = quote_handles[("BBSW6M", basis_tenor)].value()
leg3 = ql.IborLeg([notional], s3, bbsw(3, H["BBSW3M"]), dc, spreads=[spread])
leg6 = ql.IborLeg([notional], s6, bbsw(6, H["BBSW6M"]), dc)
basis_swap = ql.Swap(leg3, leg6)          # pay 3M + spread, receive 6M
basis_swap.setPricingEngine(ql.DiscountingSwapEngine(H["AONIA"]))

# Rates = every AONIA quote; basis = the basis-swap quotes of one curve (the BBSW fixings stay put).
GROUPS = {"rates": quotes.curve == "AONIA",
          "bob": quotes.instrument == "AONIA/BBSW basis",
          "b36": quotes.instrument == "3s6s basis"}

def bump(group, bp):
    for r in quotes[GROUPS[group]].itertuples():
        q = quote_handles[(r.curve, r.tenor)]
        q.setValue(q.value() + bp / 1e4)

def sensitivity(group):
    bump(group, +1); up = basis_swap.NPV()
    bump(group, -2); down = basis_swap.NPV()
    bump(group, +1)
    return (up - down) / 2

risk = {"npv": basis_swap.NPV(), "rates": sensitivity("rates"), "bob": sensitivity("bob"), "b36": sensitivity("b36")}
risk["notional_for_40k"] = 40_000 / abs(risk["b36"]) * notional
out["basis_risk"] = risk
pd.Series(risk)

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")